In [8]:
import os, re, json
import unicodedata

# Thư mục chứa các file .md
SOURCE_DIR = "../../docs/vinaya-vi/kd/mv"

# Danh sách file cần xử lý. Mỗi phần tử là 1 dict:
#   "filename": tên file .md
#   "range": (số_đầu, số_cuối) các kinh (sutta) mà file này chứa — CHỈ cần khai khi
#            nhiều file dùng chung 1 top-index (kiểu AN). Nếu 1 file = 1 kinh (MN, TMC-SN),
#            bỏ qua "range" (hoặc để None).
#   "key": ghi đè top-index thủ công — dùng khi tên file KHÔNG theo đúng quy ước
#          "prefix-SỐ-..." (số ngay sau tên nikaya = index thật). Ví dụ KN:
#          "kn-001-tap-1-..." thì "001" chỉ là số thứ tự file chạy tuần tự, còn
#          index thật (Tập mấy) nằm ở "tap-1" — phải khai "key": "1" thủ công.
#          Nếu không khai, script tự suy ra số ngay sau nikaya code trong tên file.
#
# Ví dụ MN / TMC-SN (1 file = 1 kinh, tên file chuẩn):
#   {"filename": "mn-007-the-simile-of-the-cloth.md"}
#
# Ví dụ AN (nhiều file / 1 nipata, tên file chuẩn):
#   {"filename": "an-01-001-pham-sac.md", "range": (1, 10)}
#   {"filename": "an-01-002-pham-doan-trien-cai.md", "range": (11, 20)}
#
# Ví dụ KN (tên file không chuẩn, cần "key" thủ công):
#   {"filename": "kn-001-tap-1-kinh-tieu-tung.md", "key": "1"}

FILES = [
{"filename": "pli-tv-kd-1-mahakhandhaka.md","key": "1"},
{"filename": "pli-tv-kd-2-uposathakkhandhaka.md","key": "2"},
{"filename": "pli-tv-kd-3-vassupanayikakkhandhaka.md","key": "3"},
{"filename": "pli-tv-kd-4-pavaranakkhandhaka.md","key": "4"},
{"filename": "pli-tv-kd-5-cammakkhandhaka.md","key": "5"},
{"filename": "pli-tv-kd-6-bhesajjakkhandhaka.md","key": "6"},
{"filename": "pli-tv-kd-7-kathinakkhandhaka.md","key": "7"},
{"filename": "pli-tv-kd-8-civarakkhandhaka.md","key": "8"},
{"filename": "pli-tv-kd-9-campeyyakkhandhaka.md","key": "9"},
{"filename": "pli-tv-kd-10-kosambakakkhandhaka.md","key": "10"},
]

# (Tùy chọn) đặt tiêu đề tay cho 1 top-index thay vì lấy H1 của file đầu tiên gặp.
# Hữu ích cho case gộp nhiều file (AN) vì H1 của từng phẩm không đại diện cho cả nipata.
NIPATA_TITLES = {
    # "1": "AN 1. Chương Một Pháp",
}

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 2


## 2. Các hàm xử lý

In [9]:
TOP_INDEX_RE = re.compile(r'^[a-z]+-0*(\d+)')
H1_RE = re.compile(r'^#\s+(.*)$', re.MULTILINE)
CHILD_RE = re.compile(r'^#{%d}\s+(.*)$' % CHILD_HEADING_LEVEL)
NUM_PREFIX_RE = re.compile(r'^(\d+(?:\.\d+)*)\.?\s*')

from util import  slugify


def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def insert_path(item, path, default_slug, slug=None, anchor=None):
    """Đi xuống item['children'][...] theo path (list số dạng string), tạo node
    nếu chưa có. Ở node lá: chỉ gắn slug nếu KHÁC trang mặc định của item (tránh
    lặp thừa khi con nằm cùng trang cha)."""
    node = item
    for i, k in enumerate(path):
        node.setdefault("children", {})
        node["children"].setdefault(k, {})
        node = node["children"][k]
        if i == len(path) - 1:
            if slug and slug != default_slug:
                node["slug"] = slug
            if anchor:
                node["anchor"] = anchor


ROMAN_PREFIX_RE = re.compile(r'^\*{0,2}\(([IVXLCDM]+)\)\s*', re.IGNORECASE)

_ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}

def roman_to_int(s):
    s = s.upper()
    total = 0
    prev = 0
    for ch in reversed(s):
        val = _ROMAN_VALUES.get(ch)
        if val is None:
            return None
        total += val if val >= prev else -val
        prev = max(prev, val)
    return total or None

def process_file(dirpath, spec, items, warnings, titles_override):
    filename = spec["filename"]
    file_range = spec.get("range")  # (start, end) hoặc None
    key_override = spec.get("key")  # ghi đè top-index, dùng khi tên file không theo
                                     # đúng quy ước "prefix-SỐ-..." (vd KN: "kn-001-tap-1-...")

    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    if key_override is not None:
        key = str(key_override)
    else:
        top_index = top_index_from_slug(slug)
        if top_index is None:
            warnings.append(f"{filename}: không suy ra được số thứ tự (top index) từ tên file, bỏ qua — có thể cần khai \"key\" thủ công")
            return
        key = str(int(top_index))

    is_new_key = key not in items
    item = items.setdefault(key, {})

    if is_new_key:
        if key in titles_override:
            item["title"] = titles_override[key]
        else:
            h1 = extract_h1(text)
            item["title"] = h1 if h1 else f"??? (chưa có tiêu đề cho key {key})"
            if not h1:
                warnings.append(f"{filename}: không tìm thấy H1, cần điền title tay cho key {key}")
        # Trang mặc định khi user chỉ gõ tới top-index (vd "an 1"), không có số con.
        # Với case gộp nhiều file, đây là trang của file ĐẦU TIÊN gặp trong danh sách FILES.
        item["slug"] = slug
    elif key in titles_override and item.get("title") != titles_override[key]:
        item["title"] = titles_override[key]

    default_slug = item["slug"]

    # 1) điền mặc định theo range khai báo -> đảm bảo MỌI số trong range có link,
    #    kể cả khi trong file không có heading riêng cho từng kinh
    if file_range:
        start, end = file_range
        for n in range(start, end + 1):
            insert_path(item, [str(n)], default_slug, slug=slug)

    # 2) quét heading đánh số trong file để bổ sung anchor chính xác (nếu có)
    for line in text.splitlines():
        m = CHILD_RE.match(line)
        if not m:
            continue
        heading_text = m.group(1).strip()

        num_match = NUM_PREFIX_RE.match(heading_text)
        if num_match:
            numbers = num_match.group(1).split(".")
        else:
            # Heading đánh số bằng chữ số La Mã trong ngoặc, kiểu "(IV) Kinh ..."
            # (có thể có ** bold bao ngoài) -> coi như số con phẳng, tương đương "4. Kinh ..."
            roman_match = ROMAN_PREFIX_RE.match(heading_text)
            if not roman_match:
                continue
            value = roman_to_int(roman_match.group(1))
            if value is None:
                warnings.append(
                    f'{filename}: heading "{heading_text}" có số La Mã không hợp lệ, bỏ qua'
                )
                continue
            numbers = [str(value)]

        if len(numbers) == 1:
            # Heading đánh số phẳng (kể cả số La Mã vừa đổi): số này CHÍNH LÀ
            # chỉ số con, không phải số lặp lại của top-index.
            path = numbers
        else:
            # Heading đánh số kép kiểu "2.2 Dutiyakassapasutta" (SN, AN...): số đầu
            # phải trùng top-index của file, phần còn lại mới là chỉ số con.
            if numbers[0] != key:
                warnings.append(
                    f'{filename}: heading "{heading_text}" có số đầu ({numbers[0]}) khác key ({key})'
                )
                continue
            path = numbers[1:]
        if not path:
            continue
        anchor = slugify(heading_text)
        insert_path(item, path, default_slug, slug=slug, anchor=anchor)


## 3. Chạy xử lý

In [10]:
items = {}
warnings = []

for spec in FILES:
    process_file(SOURCE_DIR, spec, items, warnings, NIPATA_TITLES)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


Không có cảnh báo.


## 4. Kết quả — dán vào `quicklink-data.js`

Các trường `"???"` là chỗ bạn tự điền (`folder`, edition key, `label`, `path`, `index_length`).

In [11]:
"""
- [01. Tiểu Tụng (Kp - Khuddakapāṭha)](/kinhtieubo/pali/kp/) done
- [02. Pháp Cú (Dhp - Dhammapada)](/kinhtieubo/pali/dhp/) done
- [03. Phật Tự Thuyết (Ud - Udāna)](/kinhtieubo/pali/ud/) done
- [04. Phật Thuyết Như Vậy (Iti - Itivuttaka)](/kinhtieubo/pali/iti/) done
- [05. Kinh Tập (Sn - Suttanipāta)](/kinhtieubo/pali/snp/) done
- [06. Chuyện Thiên Cung (Vv - Vimānavatthu)](/kinhtieubo/pali/vv/)
- [07. Chuyện Ngạ Quỷ (Pv - Petavatthu)](/kinhtieubo/pali/pv/)
- [08. Trưởng Lão Tăng Kệ (Thag - Theragāthā)](/kinhtieubo/pali/thag/) done
- [09. Trưởng Lão Ni Kệ (Thig - Therīgāthā)](/kinhtieubo/pali/thig/) done
- [10. Thánh Nhân Ký Sự - Phần Tăng (Tha-ap - Apadāna I)](/kinhtieubo/pali/tha-ap/)
- [11. Thánh Nhân Ký Sự - Phần Ni (Thi-ap - Apadāna II)](/kinhtieubo/pali/thi-ap/)
- [12. Phật Sử (Bv - Buddhavaṃsa)](/kinhtieubo/pali/bv/)
- [13. Hạnh Tạng (Cp - Cariyāpiṭaka)](/kinhtieubo/pali/cp/)
- [14. Bổn Sanh (Ja - Jātaka)](/kinhtieubo/pali/ja/)
- [15. Đại Diễn Giải (Mnd - Mahāniddesa)](/kinhtieubo/pali/mnd/)
- [16. Tiểu Diễn Giải (Cnd - Cūḷaniddesa)](/kinhtieubo/pali/cnd/)
- [17. Phân Tích Đạo (Ps - Paṭisambhidāmagga)](/kinhtieubo/pali/ps/)
- [18. Đạo Luận (Ne - Nettippakaraṇa)](/kinhtieubo/pali/ne/)
- [19. Mi Tiên Vấn Đáp (Mil - Milindapañhā)](/kinhtieubo/pali/mil/)
- [20. Chú Thích Kinh Tạng (Pe - Peṭakopadesa)](/kinhtieubo/pali/pe/)

Hướng dẫn vào link-kn.js trong đó có mẫu sẫn kp,
ví dụ làm dhp:
1. điền tên file(s) "kn-002-tap-2-kinh-phap-cu.md"
2. pháp cú là Dhp thì thêm `Dhp: [],` vào file link-kn.js
3. thay [] với dữ liệu sinh ra theo output
"""

output = {
    "folder": "vinaya-vi",
    "editions": {
        "tmc": {
            "label": "pali (Việt)",
            "path": "kd/mv",
            "index_length": 2,
            "items": items,
        }
    },
}

print(json.dumps(output, ensure_ascii=False, indent=2))


{
  "folder": "vinaya-vi",
  "editions": {
    "tmc": {
      "label": "pali (Việt)",
      "path": "kd/mv",
      "index_length": 2,
      "items": {
        "1": {
          "title": "PLI-TV-KD 1. CHƯƠNG LỚN",
          "slug": "pli-tv-kd-1-mahakhandhaka",
          "children": {
            "1": {
              "anchor": "_1-chuyen-ve-cay-bo-de"
            },
            "2": {
              "anchor": "_2-chuyen-ve-cay-ajapala"
            },
            "3": {
              "anchor": "_3-chuyen-ve-ran-mucalinda"
            },
            "4": {
              "anchor": "_4-chuyen-ve-cay-rajayatana"
            },
            "5": {
              "anchor": "_5-chuyen-pham-thien-thinh-cau"
            },
            "6": {
              "anchor": "_6-chuyen-nhom-nam-vi"
            },
            "7": {
              "anchor": "_7-chuyen-xuat-gia"
            },
            "8": {
              "anchor": "_8-chuyen-ve-ac-ma"
            },
            "9": {
              "anchor": 

## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
